# Step 2: Automatic Initial Landmark Search

Automatically find initial seed landmarks (Pt-0…Pt-5) aligning CZ with HCR, replacing manual BigWarp placement.

**Approach A** — Constellation selection + Hough XY voting  
**Approach B** — Rotation search + 3D density cross-correlation (full-cloud)

Output: `{subject}_{date}_landmarks_initial.csv`

Validation: precision and recall against ground-truth `coreg_table.csv` using direct cell-ID lookup.

In [ ]:
import sys, json, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull

sys.path.insert(0, str(Path('.')))

from coreg_data_loading import (
    find_coreg_dirs, parse_coreg_dir_name, load_subject_data,
    save_landmarks,
)
from coreg_alignment import (
    CZ_RESOLUTION_UM,
    apply_rigid, rotation_search,
    extract_seed_landmarks_zxy,
    robust_tissue_bounds, select_constellation,
    match_constellation, verify_constellation_match,
)

%load_ext autoreload
%autoreload 2

DATA_DIR    = Path('/root/capsule/data')
SCRATCH_DIR = Path('/scratch')
TEMPLATE_PATH = DATA_DIR / 'coreg_transform_template.json'

In [ ]:
# ── Parameters ───────────────────────────────────────────────────────────────
SUBJECT_ID   = '790322'
CZSTACK_DATE = '2025-07-10'

# GFP threshold
GFP_THRESHOLD_COUNTS = 5

# Approach A — constellation
CONSTEL_DZ_UM      = 15.0   # initial z-slab above surface (µm)
CONSTEL_DZ_MAX_UM  = 60.0   # max z-slab depth to try
CONSTEL_N_CELLS    = 6      # target constellation size
MIP_W_UM           = 55.0   # HCR z-window half-width for Hough (µm)
Z_STEP_UM          = 25.0   # step between z-center candidates (µm)
HOUGH_BIN_UM       = 5.0    # Hough accumulator bin size (µm)
MATCH_THRESH_UM    = 15.0   # XY match threshold for scoring (µm)
VERIFY_3D_UM       = 25.0   # 3D distance threshold for verification (µm)
VERIFY_CONSIST_UM  = 20.0   # pairwise distance consistency (µm)

# Seed extraction after alignment
SEED_Z_WINDOW_UM   = 150.0  # Z half-window for ZXY seed extraction (µm)
SEED_XY_THRESH_UM  = 22.0   # XY MNN radius for seed extraction (µm)

# Approach B — rotation search
DENSITY_VOXEL_UM   = 5.0
N_REFINE           = 3

# Load transform template
if TEMPLATE_PATH.exists():
    with open(TEMPLATE_PATH) as f:
        template = json.load(f)
    print('Loaded transform template:', json.dumps(template, indent=2))
else:
    print('WARNING: template not found, using defaults')
    template = {
        'z_rotation_range_deg': [-15, 15],
        'pitch_range_deg':      [-186, 186],
        'roll_range_deg':       [-15, 15],
    }

In [ ]:
# ── Load data ────────────────────────────────────────────────────────────────
coreg_dirs = find_coreg_dirs(DATA_DIR)
coreg_dir  = None
for d in coreg_dirs:
    sid, cdate = parse_coreg_dir_name(d)
    if sid == SUBJECT_ID and cdate == CZSTACK_DATE:
        coreg_dir = d
        break

save_dir = SCRATCH_DIR / f'{SUBJECT_ID}_{CZSTACK_DATE}_step2'
save_dir.mkdir(parents=True, exist_ok=True)
print(f'Coreg dir : {coreg_dir}')
print(f'Save dir  : {save_dir}')

data = load_subject_data(coreg_dir, SUBJECT_ID, CZSTACK_DATE,
                         gfp_threshold=GFP_THRESHOLD_COUNTS)
czstack_df = data['czstack_df']
hcr_df     = data['hcr_df']
hcr_scales = data['hcr_scales']
spot_counts = data['spot_counts']

gfp_mask   = spot_counts[spot_counts['is_gfp'] == True]['hcr_cell_id']
hcr_gfp_df = hcr_df[hcr_df['hcr_cell_id'].isin(gfp_mask)].reset_index(drop=True)

print(f'CZ cells        : {len(czstack_df)}')
print(f'HCR cells total : {len(hcr_df)}')
print(f'HCR GFP+ cells  : {len(hcr_gfp_df)}  (threshold={GFP_THRESHOLD_COUNTS})')
print(f'HCR scales µm/px: z={hcr_scales["scale_z"]:.4f}  '
      f'y={hcr_scales["scale_y"]:.4f}  x={hcr_scales["scale_x"]:.4f}')

In [ ]:
# ── Convert to µm ────────────────────────────────────────────────────────────
hcr_res = np.array([hcr_scales['scale_z'], hcr_scales['scale_y'], hcr_scales['scale_x']])

# CZ centroids in µm (z, y, x)
cz_um = czstack_df[['czstack_z', 'czstack_y', 'czstack_x']].values * CZ_RESOLUTION_UM

# DataFrame with µm coords (keep czstack_cell_id); used by select_constellation
cz_df_um = czstack_df[['czstack_cell_id']].copy()
cz_df_um['czstack_z'] = cz_um[:, 0]
cz_df_um['czstack_y'] = cz_um[:, 1]
cz_df_um['czstack_x'] = cz_um[:, 2]

# HCR GFP+ centroids in µm
hcr_gfp_um = hcr_gfp_df[['hcr_z', 'hcr_y', 'hcr_x']].values * hcr_res

print(f'CZ µm range:  z=[{cz_um[:,0].min():.0f}, {cz_um[:,0].max():.0f}]  '
      f'y=[{cz_um[:,1].min():.0f}, {cz_um[:,1].max():.0f}]  '
      f'x=[{cz_um[:,2].min():.0f}, {cz_um[:,2].max():.0f}]')
print(f'HCR GFP+ µm:  z=[{hcr_gfp_um[:,0].min():.0f}, {hcr_gfp_um[:,0].max():.0f}]  '
      f'y=[{hcr_gfp_um[:,1].min():.0f}, {hcr_gfp_um[:,1].max():.0f}]  '
      f'x=[{hcr_gfp_um[:,2].min():.0f}, {hcr_gfp_um[:,2].max():.0f}]')

## Approach A — Constellation + Hough XY Voting

1. Find CZ tissue surface (`robust_tissue_bounds`)
2. Select k ≤ 6 near-surface cells whose XY convex hull contains zero other CZ cells (`select_constellation`)
3. For each rotation candidate × HCR z-window, run Hough XY vote to find translation (`match_constellation`)
4. Verify top-5 hypotheses geometrically (`verify_constellation_match`)
5. Apply best hypothesis to all CZ cells → extract seed landmarks via ZXY MNN

In [ ]:
# ── A1: tissue surface + constellation selection ──────────────────────────────
cz_z_surface, cz_z_top, cz_thickness = robust_tissue_bounds(
    cz_df_um['czstack_z'].values,
    cz_df_um['czstack_y'].values,
    cz_df_um['czstack_x'].values,
)
print(f'CZ tissue: surface={cz_z_surface:.1f} µm  top={cz_z_top:.1f} µm  '
      f'thickness={cz_thickness:.1f} µm')

constellation_df = select_constellation(
    cz_df_um, cz_z_surface,
    dz_um=CONSTEL_DZ_UM, dz_max_um=CONSTEL_DZ_MAX_UM,
    n_cells=CONSTEL_N_CELLS,
    verbose=True,
)

if constellation_df.empty:
    print('ERROR: no constellation found — check dz_max_um or n_cells')
else:
    print(f'\nConstellation ({len(constellation_df)} cells):')
    print(constellation_df[['czstack_cell_id', 'czstack_z', 'czstack_y', 'czstack_x', 'hull_intruders']])

# Plot constellation in CZ XY
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(cz_df_um['czstack_x'], cz_df_um['czstack_y'],
           s=3, alpha=0.3, c='steelblue', label='All CZ cells')
if not constellation_df.empty:
    cx = constellation_df['czstack_x'].values
    cy = constellation_df['czstack_y'].values
    ax.scatter(cx, cy, s=60, c='red', zorder=5, label='Constellation')
    for i, row in constellation_df.iterrows():
        ax.annotate(str(int(row['czstack_cell_id'])),
                    (row['czstack_x'], row['czstack_y']),
                    fontsize=8, color='darkred')
    # Draw hull
    if len(constellation_df) >= 3:
        pts = np.stack([cy, cx], axis=1)
        try:
            hull = ConvexHull(pts)
            for simplex in hull.simplices:
                ax.plot(pts[simplex, 1], pts[simplex, 0], 'r-', lw=1.5, alpha=0.7)
        except Exception:
            pass
ax.set_xlabel('CZ x (µm)'); ax.set_ylabel('CZ y (µm)')
ax.set_title(f'{SUBJECT_ID}: Constellation in CZ XY')
ax.legend()
plt.tight_layout()
plt.savefig(save_dir / f'{SUBJECT_ID}_constellation.png', dpi=100)
plt.show()

In [ ]:
# ── A2: match constellation in HCR ───────────────────────────────────────────
assert not constellation_df.empty, 'No constellation — cannot proceed'

constellation_um = constellation_df[['czstack_z', 'czstack_y', 'czstack_x']].values

hypotheses = match_constellation(
    constellation_um, hcr_gfp_um, template,
    W_um=MIP_W_UM,
    z_step_um=Z_STEP_UM,
    match_threshold_um=MATCH_THRESH_UM,
    bin_size_um=HOUGH_BIN_UM,
    verbose=True,
)

print(f'\nTop-{len(hypotheses)} hypotheses:')
for i, h in enumerate(hypotheses):
    print(f'  [{i}] score={h["score"]}/{len(constellation_um)}  '
          f'z_center={h["z_center"]:.0f} µm  '
          f't=({h["t_y"]:.0f}, {h["t_x"]:.0f}) µm  '
          f'peak_count={h["peak_count"]}')

In [ ]:
# ── A3: verify hypotheses and pick best ──────────────────────────────────────
best_hyp_A = None
for i, hyp in enumerate(hypotheses):
    ok = verify_constellation_match(
        hyp, constellation_um, hcr_gfp_um,
        dist_threshold_3d_um=VERIFY_3D_UM,
        dist_consistency_um=VERIFY_CONSIST_UM,
    )
    print(f'Hypothesis [{i}]: verification={"PASS" if ok else "FAIL"}')
    if ok and best_hyp_A is None:
        best_hyp_A = hyp

if best_hyp_A is None:
    print('\nWARNING: no hypothesis passed verification — relaxing thresholds')
    # Fallback: accept best-scoring without strict verification
    best_hyp_A = hypotheses[0] if hypotheses else None

if best_hyp_A is not None:
    R_A  = best_hyp_A['R']
    # Estimate full 3D translation from the hypothesis
    c_rot = (R_A @ constellation_um.T).T
    tz_A  = best_hyp_A['z_center'] - float(c_rot[:, 0].mean())
    t_A   = np.array([tz_A, best_hyp_A['t_y'], best_hyp_A['t_x']])
    print(f'\nBest hypothesis A: t={t_A.round(1)} µm, '
          f'z_center={best_hyp_A["z_center"]:.0f} µm')

    # Apply to all CZ cells
    P_aligned_A = apply_rigid(cz_um, R_A, t_A)
    print(f'P_aligned_A range: '
          f'z=[{P_aligned_A[:,0].min():.0f}, {P_aligned_A[:,0].max():.0f}]  '
          f'y=[{P_aligned_A[:,1].min():.0f}, {P_aligned_A[:,1].max():.0f}]  '
          f'x=[{P_aligned_A[:,2].min():.0f}, {P_aligned_A[:,2].max():.0f}]')
else:
    print('ERROR: no valid hypothesis — Approach A failed')
    P_aligned_A = None

In [ ]:
# ── A4: extract seed landmarks from Approach A alignment ─────────────────────
if P_aligned_A is not None:
    seeds_A = extract_seed_landmarks_zxy(
        P_aligned_A, czstack_df,
        hcr_gfp_um, hcr_gfp_df,
        z_window_um=SEED_Z_WINDOW_UM,
        xy_threshold_um=SEED_XY_THRESH_UM,
    )
    print(f'Approach A seed landmarks: {len(seeds_A)}')

    if len(seeds_A) > 0:
        save_path_A = save_dir / f'{SUBJECT_ID}_{CZSTACK_DATE}_landmarks_A.csv'
        save_landmarks(seeds_A, save_path_A)
        print(f'Saved → {save_path_A}')
        print(seeds_A.head())

        # Plot
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        for ax, (dim_y, dim_x, label) in zip(
            axes,
            [(1, 2, 'Y-X'), (0, 2, 'Z-X')]
        ):
            ax.scatter(hcr_gfp_um[:, dim_x], hcr_gfp_um[:, dim_y],
                       s=2, alpha=0.2, c='royalblue', label='HCR GFP+')
            ax.scatter(P_aligned_A[:, dim_x], P_aligned_A[:, dim_y],
                       s=5, alpha=0.4, c='gray', label='CZ aligned')
            ax.scatter(
                seeds_A['hcr_x'].values * hcr_res[2],
                seeds_A['hcr_y'].values * hcr_res[1] if dim_y == 1 else
                seeds_A['hcr_z'].values * hcr_res[0],
                s=40, c='orangered', zorder=5, label=f'Seeds A (n={len(seeds_A)})')
            ax.set_xlabel(f'{label[2]} (µm)'); ax.set_ylabel(f'{label[0]} (µm)')
            ax.set_title(f'{label} — Approach A seeds')
            ax.legend(markerscale=2)
        plt.suptitle(f'Approach A: {SUBJECT_ID}')
        plt.tight_layout()
        plt.savefig(save_dir / f'{SUBJECT_ID}_seeds_A.png', dpi=100)
        plt.show()
    else:
        print('WARNING: no seeds extracted from Approach A alignment')
        seeds_A = pd.DataFrame()
else:
    seeds_A = pd.DataFrame()

## Approach B — Rotation Search + 3D Density Cross-Correlation

Full-cloud approach: enumerates ~125 rotation candidates, builds Gaussian density volumes for CZ and HCR, uses 3D FFT cross-correlation to find translation, then refines with Nelder-Mead.

In [ ]:
# ── Approach B: rotation search ───────────────────────────────────────────────
R_B, t_B, score_B = rotation_search(
    cz_um, hcr_gfp_um,
    template=template,
    voxel_um=DENSITY_VOXEL_UM,
    threshold_um=SEED_XY_THRESH_UM,
    n_refine=N_REFINE,
    verbose=True,
)

print(f'\nApproach B best: score={score_B}/{len(cz_um)} ({100*score_B/len(cz_um):.1f}%)')
P_aligned_B = apply_rigid(cz_um, R_B, t_B)

seeds_B = extract_seed_landmarks_zxy(
    P_aligned_B, czstack_df,
    hcr_gfp_um, hcr_gfp_df,
    z_window_um=SEED_Z_WINDOW_UM,
    xy_threshold_um=SEED_XY_THRESH_UM,
)
print(f'Approach B seed landmarks: {len(seeds_B)}')

if len(seeds_B) > 0:
    save_path_B = save_dir / f'{SUBJECT_ID}_{CZSTACK_DATE}_landmarks_B.csv'
    save_landmarks(seeds_B, save_path_B)
    print(f'Saved → {save_path_B}')

# Quick overlay plot
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(hcr_gfp_um[:, 2], hcr_gfp_um[:, 1], s=2, alpha=0.2, c='royalblue', label='HCR GFP+')
ax.scatter(P_aligned_B[:, 2], P_aligned_B[:, 1], s=5, alpha=0.5, c='tomato', label='CZ aligned (B)')
ax.set_xlabel('x (µm)'); ax.set_ylabel('y (µm)')
ax.set_title(f'Approach B XY overlay: {SUBJECT_ID} | score={score_B}/{len(cz_um)}')
ax.legend(markerscale=2)
plt.tight_layout()
plt.savefig(save_dir / f'{SUBJECT_ID}_alignment_B.png', dpi=100)
plt.show()

## Validation

Parse `ids` column (`cz{N}-hcr{M}`) from each seed set and check against `coreg_table.csv` (columns `czstack_id`, `hcr_id`).  
A seed is **correct** if the (czstack_id, hcr_id) pair exists in the coreg table.

In [ ]:
# ── Load ground-truth coreg table ────────────────────────────────────────────
def find_coreg_table(coreg_dir, subject_id):
    """Find the coreg_table.csv for this subject."""
    if coreg_dir is None:
        return None
    for cand in sorted(coreg_dir.glob('*coreg_table*.csv')):
        return cand
    return None

gt_path = find_coreg_table(coreg_dir, SUBJECT_ID)
print(f'Ground-truth table: {gt_path}')

if gt_path is not None and gt_path.exists():
    gt_df = pd.read_csv(gt_path)
    print(f'GT rows: {len(gt_df)}  columns: {list(gt_df.columns)}')
    gt_pairs = set(zip(gt_df['czstack_id'].astype(int), gt_df['hcr_id'].astype(int)))
    print(f'GT pairs: {len(gt_pairs)}')
else:
    gt_pairs = None
    print('WARNING: no coreg_table found — skipping validation')


def validate_seeds(seeds_df, gt_pairs, label):
    """Parse ids column and compute precision against ground truth."""
    if seeds_df is None or seeds_df.empty:
        print(f'{label}: no seeds')
        return None
    if gt_pairs is None:
        print(f'{label}: {len(seeds_df)} seeds (no GT to validate against)')
        return None

    seeds_df = seeds_df.copy()
    seeds_df['czstack_cell_id'] = seeds_df['ids'].str.extract(r'cz(\d+)').astype(int)
    seeds_df['hcr_cell_id']     = seeds_df['ids'].str.extract(r'hcr(\d+)').astype(int)
    seeds_df['correct'] = seeds_df.apply(
        lambda r: (int(r['czstack_cell_id']), int(r['hcr_cell_id'])) in gt_pairs, axis=1
    )

    n_correct  = seeds_df['correct'].sum()
    n_total    = len(seeds_df)
    precision  = n_correct / n_total if n_total > 0 else 0.0
    recall_denom = len(gt_pairs)
    recall     = n_correct / recall_denom if recall_denom > 0 else 0.0

    print(f'{label}: {n_total} seeds  correct={n_correct}  '
          f'precision={precision:.3f}  recall={recall:.3f}')
    return seeds_df


val_A = validate_seeds(seeds_A, gt_pairs, 'Approach A')
val_B = validate_seeds(seeds_B, gt_pairs, 'Approach B')

In [ ]:
# ── Validation plots ──────────────────────────────────────────────────────────
for val_df, label in [(val_A, 'Approach A'), (val_B, 'Approach B')]:
    if val_df is None or val_df.empty:
        continue
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.scatter(hcr_gfp_um[:, 2], hcr_gfp_um[:, 1],
               s=2, alpha=0.15, c='steelblue', label='HCR GFP+')
    correct   = val_df[val_df['correct']]
    incorrect = val_df[~val_df['correct']]
    if len(correct):
        ax.scatter(
            correct['hcr_x'].values * hcr_res[2],
            correct['hcr_y'].values * hcr_res[1],
            s=40, c='limegreen', zorder=5, label=f'Correct ({len(correct)})')
    if len(incorrect):
        ax.scatter(
            incorrect['hcr_x'].values * hcr_res[2],
            incorrect['hcr_y'].values * hcr_res[1],
            s=40, c='crimson', marker='x', zorder=5, label=f'Incorrect ({len(incorrect)})')
    prec = len(correct) / len(val_df) if len(val_df) > 0 else 0
    ax.set_xlabel('HCR x (µm)'); ax.set_ylabel('HCR y (µm)')
    ax.set_title(f'{label}: {SUBJECT_ID}  precision={prec:.2f}')
    ax.legend(markerscale=2)
    plt.tight_layout()
    lbl = 'A' if 'A' in label else 'B'
    plt.savefig(save_dir / f'{SUBJECT_ID}_validation_{lbl}.png', dpi=100)
    plt.show()

# ── Select and save best ──────────────────────────────────────────────────────
def precision_of(val_df):
    if val_df is None or val_df.empty:
        return 0.0
    return val_df['correct'].mean() if 'correct' in val_df.columns else 0.0

def n_seeds_of(df):
    return len(df) if df is not None else 0

# Prefer whichever has higher precision; break ties by n_seeds
prec_A = precision_of(val_A)
prec_B = precision_of(val_B)
n_A    = n_seeds_of(seeds_A)
n_B    = n_seeds_of(seeds_B)

print(f'\nApproach A: precision={prec_A:.3f}  n={n_A}')
print(f'Approach B: precision={prec_B:.3f}  n={n_B}')

if prec_A >= prec_B and n_A >= 1:
    best_seeds = seeds_A
    best_label = 'A'
elif n_B >= 1:
    best_seeds = seeds_B
    best_label = 'B'
else:
    best_seeds = pd.DataFrame()
    best_label = 'none'

print(f'\nSelected: Approach {best_label}')

if not best_seeds.empty:
    best_path = save_dir / f'{SUBJECT_ID}_{CZSTACK_DATE}_landmarks_initial.csv'
    save_landmarks(best_seeds, best_path)
    print(f'Saved best landmarks → {best_path}')
    print(f'n_seeds={len(best_seeds)}  (use as input to step_3)')